In [ ]:
## This script extracts deep features from LFDP patches

In [ ]:
# import packages
import os, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
## packages for deep learning
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

## check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
# load shadow-free image
img = cv2.imread('../output/images/shadow_free.jpg')
## transform color channel
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# load the pretrained cnn model ResNet-50
base_model = models.resnet50(pretrained=True)
# 移除最后的全连接层，只保留到avgpool
resnet_model = nn.Sequential(*list(base_model.children())[:-1])
resnet_model = resnet_model.to(device)
resnet_model.eval()

# 定义预处理（等价于Keras的preprocess_input）
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                       std=[0.229, 0.224, 0.225])
])

# extract features, and save in the feat_cnn.csv file
## set resolution and window size
rs = 100 ## resolution 
size_wind = 100 # window size 
num_row = int(img.shape[0]/rs)
num_col = int(img.shape[1]/rs)
## extract features
csvPath = '../output/features/feat_cnn.csv'
f = open(csvPath, 'w')
print('...Start Feature Extraction...')
time_start = datetime.now()

with torch.no_grad():  # 不计算梯度
    for i in range(num_row):
        if (i+1) % 20 == 0:
            print('Finished:', "{:.1%}".format((i+1) / num_row), end="\r")
        
        # 准备一行的所有patches
        batch_tensors = []
        for j in range(num_col):
            r_1 = int(np.max([0, i*rs + rs/2 - size_wind/2]))
            r_2 = int(np.min([img.shape[0], i*rs + rs/2 +  size_wind/2]))
            c_1 = int(np.max([0, j*rs + rs/2 - size_wind/2]))
            c_2 = int(np.min([img.shape[1], j*rs + rs/2 + size_wind/2]))
            img_patch = img[r_1:r_2, c_1:c_2]
            img_resize = cv2.resize(img_patch, (224, 224))
            
            # 预处理转为tensor
            img_tensor = preprocess(img_resize)
            batch_tensors.append(img_tensor)
        
        # 堆叠为batch
        x_img = torch.stack(batch_tensors).to(device)  # [num_col, 3, 224, 224]
        
        # 前向传播提取特征
        preds = resnet_model(x_img)  # [num_col, 2048, 1, 1]
        preds = preds.squeeze(-1).squeeze(-1)  # [num_col, 2048]
        preds = preds.cpu().numpy()  # 转回numpy
        
        # save features line by line
        for j in range(num_col):
            str1 = list(preds[j])
            line = ', '.join(list(map(str, str1))) + '\n'
            f.write(line)        

f.close()
print('Time for feature extraction:', datetime.now()-time_start)